# 1.0 Importação das Bibliotecas Usadas e do Dataset

In [1]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

from sklearn import model_selection
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from scipy.optimize import minimize

from sklearn.metrics import f1_score
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

from sklearn.ensemble import AdaBoostClassifier

In [2]:
data = pd.read_csv("dataset_tratado.csv")
data

,Unnamed: 0,Age,CDR,SES,nWBV,Visit,MMSE,ASF,eTIV,Group,MR Delay
0,0,87.0,0.0,2.0,0.696106,1.0,27.0,0.883440,1986.550000,0.0,0.0
1,1,88.0,0.0,2.0,0.681062,2.0,30.0,0.875539,2004.479526,0.0,457.0
2,2,75.0,0.5,1.8,0.736336,1.0,23.0,1.045710,1678.290000,1.0,0.0
3,3,76.0,0.5,1.6,0.713402,2.0,28.0,1.010000,1737.620000,1.0,560.0
4,4,80.0,0.5,2.6,0.701236,3.0,22.0,1.033623,1697.911134,1.0,1895.0
...,...,...,...,...,...,...,...,...,...,...,...
368,368,82.0,0.5,1.0,0.693926,2.0,28.0,1.036690,1692.880000,1.0,842.0
369,369,86.0,0.5,1.0,0.675457,3.0,26.0,1.039686,1688.009649,1.0,2297.0
370,370,61.0,0.0,2.0,0.801006,1.0,30.0,1.330540,1319.020000,0.0,0.0
371,371,63.0,0.0,2.0,0.795981,2.0,30.0,1.322890,1326.650000,0.0,763.0


# 2.0 Pre-processamento dos Dados

## 2.1 Criando X e y

In [3]:
X = np.array(data.drop(['Group', 'Unnamed: 0'], axis=1))
y = np.array(data['Group'])

## 2.2 Separando em Treino e Teste

In [4]:
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, random_state=RANDOM_STATE, test_size=0.2)

# 3.0 Modelagem dos Dados

In [5]:
def Modelagem(X_train, y_train, X_test, Modelo):
    
    pca_pipeline = Pipeline([
        ('scaler', StandardScaler()),        
        ('pca', PCA(n_components=0.95)),      
        ('modelo', Modelo)   
    ])
    
    pca_pipeline.fit(X_train, y_train)
    
    previsoes = pca_pipeline.predict(X_test)

    return previsoes, pca_pipeline

In [6]:
def ModelagemRC(X_train, y_train, X_test, Modelo):
    
    Modelo.fit(X_train, y_train)
    
    return Modelo.predict(X_test)

In [7]:
def ModelagemRLO(X_train, y_train, X_test, Solver):
    
    pipeline = Pipeline([
        ('scaler', StandardScaler()),        # Normalização essencial para solvers de gradiente 
        ('pca', PCA(n_components=0.95)),     # Redução de dimensionalidade conforme o artigo 
        ('modelo', LogisticRegression(solver=Solver, max_iter=1000, tol=1e-5))   
    ])
    
    start_time = time.perf_counter()
    pipeline.fit(X_train, y_train)
    end_time = time.perf_counter()
    
    previsoes = pipeline.predict(X_test)
    tempo_execucao = end_time - start_time
    
    return previsoes, tempo_execucao

# 4.0 Modelos de Aprendizagem

## 4.1 Random Forest

O **Random Forest (RF)** é amplamente reconhecido como um dos métodos mais proeminentes no campo do aprendizado de máquina para o desenvolvimento de modelos preditivos em diversas disciplinas. O modelo é uma versão mais abrangente que a Árvore de Decisão (DT), pois utiliza múltiplos classificadores para alcançar maior precisão nas predições.

O RF pode ser aplicado tanto em *regressão*, onde as predições de cada árvore são calculadas pela média, quanto em *classificação*, onde a predição é feita coletando os votos da maioria dentre as classes, utilizando os votos provenientes das árvores individuais.

A equação utilizada pelo modelo para criar uma estimativa com todas as árvores é:

$$\bar{r}_n(X, D_n) = E_\theta[r_n(X, \theta, D_n)]$$

onde:

- $\bar{r}_n(X, D_n)$ é a estimativa do Random Forest para a entrada $X$ no conjunto de dados $D_n$
- $E_\theta$ denota a expectativa em relação ao parâmetro aleatório $\theta$
- $r_n(X, \theta, D_n)$ é a predição de uma árvore individual com parâmetro aleatório $\theta$
- $D_n$ é o conjunto de dados de treinamento com $n$ amostras

Treinando o modelo Random Forest utilizando os hiperparâmetros default e exibindo as métricas no conjunto de Teste

In [8]:
modelo = RandomForestClassifier(random_state=42) # inicializando o modelo com random_state = 42

random_forest_predict = ModelagemRC(X_train, y_train, X_test, modelo) # treinando o modelo e realizando o predict

print("=== Relatório de Métricas (Random Forest) ===")

print(classification_report(y_test, random_forest_predict)) # exibindo os resultados

=== Relatório de Métricas (Random Forest) ===
              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



Apesar da acurácia obtida ser semelhante à acurácia exibida no artigo, utilizamos o GridSearchCV da biblioteca scikit-learn para realizar a busca dos melhores hiperparâmetros por meio de cross validation.

In [9]:
# gera valores igualmente espaçados para testar diferentes quantidades de árvores na floresta
n_estimators = [int(x) for x in np.linspace(start=100, stop = 200, num=10)]
# define as estratégias de seleção de atributos em cada divisão da árvore
max_features = ['sqrt', 'log2']
# gera valores igualmente espaçados para controlar a profundidade máxima das árvores
max_depth = [int(x) for x in np.linspace(start=10, stop = 40, num=4)]
max_depth.append(None)
# gera valores para o número mínimo de amostras em cada folha
min_samples_leaf = [int(x) for x in np.linspace(start=1, stop = 3, num=3)]

param_grid = {
    'n_estimators' : n_estimators,
    'max_features' : max_features,
    'min_samples_leaf' : min_samples_leaf,
    'max_depth' : max_depth,
}

In [10]:
random_forest = RandomForestClassifier(random_state=42) # inicializando o modelo com random_state = 42

cv_random_forest = GridSearchCV(estimator=random_forest, param_grid=param_grid, cv = 5, verbose=2, n_jobs=-1)

cv_random_forest.fit(X_train, y_train) # realizando o gridseachcv

best_model = cv_random_forest.best_estimator_ # armazena o melhor modelo encontrado

# exibindo os resultados associado ao melhor modelo encontrado

print('Ein: %0.4f' % (1 - accuracy_score(y_train, best_model.predict(X_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test, best_model.predict(X_test))))
print(classification_report(y_test, best_model.predict(X_test)))

print("Melhor Modelo:", best_model)

Fitting 5 folds for each of 300 candidates, totalling 1500 fits
Ein: 0.0101
Eout: 0.0400
              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75

Melhor Modelo: RandomForestClassifier(max_depth=10, min_samples_leaf=2, random_state=42)


Após a execução do GridSearchCV, observamos que melhores hiperparâmetros para o RandomForestClassifier são max_depth (profundidade máxima) = 10 e min_samples_leaf (mínimo de amostras por folha) = 2.

In [11]:
modelo = RandomForestClassifier(max_depth=10, min_samples_leaf=2, random_state=42) # inicializando o modelo com os hiperparametros encontrados 

random_forest_predict = ModelagemRC(X_train, y_train, X_test, modelo) # treinando o modelo e realizando o predict

print("=== Relatório de Métricas (Random Forest) ===")

print(classification_report(y_test, random_forest_predict)) # exibindo os resultados

=== Relatório de Métricas (Random Forest) ===
              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



## 4.2 Logistic Regression

Treinando o modelo Logistic Regression e exibindo as métricas com o conjunto de Teste

In [12]:
lr = LogisticRegression(max_iter=1000, tol=1e-5)

previsoes_rl, _ = Modelagem(X_train, y_train, X_test, lr)

print("=== Relatório de Métricas (Regressão Logística) ===")

print(classification_report(y_test, previsoes_rl))

=== Relatório de Métricas (Regressão Logística) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



### 4.2.1 O Problema de Otimização

Diferente de modelos de árvore, a **Regressão Logística (LR)** é fundamentalmente um problema de otimização contínua.Nosso objetivo é encontrar o vetor de parâmetros $w$ que maximize a probabilidade de os dados observados pertencerem às suas classes reais, o que é feito através da minimização de uma função de perda.

#### 4.2.2 Função de Hipótese e Sigmoide
Para mapear qualquer valor real em um intervalo de probabilidade $[0, 1]$, utilizamos a função logística (sigmoide):

$$P(Y) = \frac{1}{1+e^{-(b_{0}+b_{1}x_{1}+b_{2}x_{2}+...+b_{n}x_{n})}}$$

Onde:
* $b_0, b_1, \dots, b_n$ são os coeficientes (pesos) que o modelo precisa aprender via otimização.
* $x_1, x_2, \dots, x_n$ representam as features do dataset OASIS, como Idade, nWBV, MMSE, entre outras.
* A saída representa se o paciente possui Alzheimer (codificado como 1) ou não (codificado como 0).



#### 4.2.3 Função de Custo: Log-Loss
Em Otimização Não-Linear, resolvemos este problema minimizando a **Negative Log-Likelihood** (ou Log-Loss). Esta função mede a discrepância entre as predições e os rótulos reais:

$$J(w) = -\frac{1}{N} \sum_{i=1}^{N} [y_i \log(h_w(x_i)) + (1 - y_i) \log(1 - h_w(x_i))]$$

#### 4.2.4 Convexidade e Solvers
Um ponto crucial para a disciplina é que a função de custo da Regressão Logística é **convexa**. Isso garante que:
* Não existem mínimos locais; qualquer mínimo encontrado pelos algoritmos é o **mínimo global**.
* Podemos utilizar métodos de otimização eficientes como **L-BFGS**, **Newton-CG** ou **Liblinear** para convergência rápida.



No artigo base, a eficiência deste processo de otimização permitiu que a Regressão Logística atingisse **96% de acurácia, precisão, sensibilidade e F1-score** , demonstrando ser um preditor tão robusto quanto o SVM e o Random Forest para a doença de Alzheimer.

#### 4.2.5 Regressão Logística com Solver LBFGS

In [13]:
previsoes_rl, tempo = ModelagemRLO(X_train, y_train, X_test, 'lbfgs')

print(f"Tempo de execução com solver L-BFGS: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística - LBFGS) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver L-BFGS: 0.0331 segundos

=== Relatório de Métricas (Regressão Logística - LBFGS) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



#### 4.2.6 Regressão Logística com Solver Liblinear

In [14]:
previsoes_rl, tempo = ModelagemRLO(X_train, y_train, X_test, 'liblinear')

print(f"Tempo de execução com solver Liblinear: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística - liblinear) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver Liblinear: 0.0540 segundos

=== Relatório de Métricas (Regressão Logística - liblinear) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



#### 4.2.7 Regressão Logística com Solver Newton-CG

In [15]:
previsoes_rl, tempo = ModelagemRLO(X_train, y_train, X_test, 'newton-cg')

print(f"Tempo de execução com solver Newton-CG: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística - Newton) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver Newton-CG: 0.0368 segundos

=== Relatório de Métricas (Regressão Logística - Newton) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



#### 4.2.8 Regressão Logística com Solver SAG

In [16]:
previsoes_rl, tempo = ModelagemRLO(X_train, y_train, X_test, 'sag')

print(f"Tempo de execução com solver SAG: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística - Gradiente Estocástico) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver SAG: 0.0664 segundos

=== Relatório de Métricas (Regressão Logística - Gradiente Estocástico) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



#### 4.2.9 Regressão Logística com Solver SAGA

In [17]:
previsoes_rl, tempo = ModelagemRLO(X_train, y_train, X_test, 'saga')

print(f"Tempo de execução com solver SAGA: {tempo:.4f} segundos\n")
print("=== Relatório de Métricas (Regressão Logística - Gradiente Estocástico Aprimorado) ===")
print(classification_report(y_test, previsoes_rl))

Tempo de execução com solver SAGA: 0.0225 segundos

=== Relatório de Métricas (Regressão Logística - Gradiente Estocástico Aprimorado) ===
              precision    recall  f1-score   support

         0.0       1.00      0.95      0.98        43
         1.0       0.94      1.00      0.97        32

    accuracy                           0.97        75
   macro avg       0.97      0.98      0.97        75
weighted avg       0.97      0.97      0.97        75



## 4.3 Support Vector Machine

In [18]:
predicao, _ = Modelagem(X_train, y_train, X_test, SVC())

print("=== Relatório de Métricas (SVM) ===")

print(classification_report(y_test, predicao))

=== Relatório de Métricas (SVM) ===
              precision    recall  f1-score   support

         0.0       0.98      0.95      0.96        43
         1.0       0.94      0.97      0.95        32

    accuracy                           0.96        75
   macro avg       0.96      0.96      0.96        75
weighted avg       0.96      0.96      0.96        75



### 4.3.1 Mudando os métodos de otimização

### Função de Loss — Squared Hinge SVM

$$L(w) = \frac{1}{2}||w||^2 + C \sum_{i=1}^{N} \max(0, 1 - y_i \cdot x_i^T w)^2$$

**Onde:**
- $w$ — vetor de pesos (o que o modelo aprende)
- $C$ — regularização: controla o trade-off entre margem e erros
- $y_i \in \{-1, +1\}$ — rótulo da amostra $i$
- $x_i$ — vetor de features da amostra $i$
- $m_i = y_i \cdot x_i^T w$ — margem da amostra $i$

---

### Gradiente

$$\nabla L(w) = w - 2C \sum_{i: m_i < 1} (1 - m_i) \cdot y_i x_i$$

**Onde:**
- O somatório considera apenas os pontos que violam a margem ($m_i < 1$)
- Pontos com $m_i \geq 1$ têm gradiente zero — já estão classificados corretamente com folga

---

### Efeito do parâmetro C

| $C$ | Comportamento |
|-----|--------------|
| $C \to 0$ | Margem larga, aceita mais erros — underfitting |
| $C = 1$ | Equilíbrio padrão |
| $C \to \infty$ | Margem estreita, tolera poucos erros — overfitting |

---

### Condição de otimalidade

O ponto $w^*$ é ótimo quando:

$$||\nabla L(w^*)|| \approx 0$$

In [19]:
y_df = data['Group']
y_df.value_counts()

Group
0.0    227
1.0    146
Name: count, dtype: int64

In [20]:
# Transformando os dados para o formato da classificação -1 e 1 para o SVM
y_df = np.where(y == 0.0, -1, 1)
scaler = StandardScaler()

X_train_, X_test_, y_train_, y_test_ = train_test_split(X, y_df, test_size=0.2, random_state=42)

X_train_ = scaler.fit_transform(X_train_)
X_test_ = scaler.transform(X_test_)

np.random.seed(42)

In [21]:
class SVMoptimizer:
    def __init__(self, C=1.0, tol=1e-5, maxiter=1000):
        self.C = C
        self.tol = tol
        self.maxiter = maxiter

    def loss(self, w, X, y):
        margem = y * (X @ w)
        hinge = np.maximum(0, 1 - margem) ** 2
        return 0.5 * np.dot(w, w) + self.C * np.sum(hinge)

    def gradient(self, w, X, y):
        margem = y * (X @ w)
        mascara = margem < 1
        diff = 1 - margem[mascara]
        return w - 2 * self.C * (diff * y[mascara]) @ X[mascara]

    def hessiana(self, w, X, y):
        margem = y * (X @ w)
        mascara = margem < 1
        X_mascara = X[mascara]
        return np.eye(len(w)) + 2 * self.C * X_mascara.T @ X_mascara

    def otimizar(self, X, y, w0, methods):
        max_iter_options = {
            'CG':        {'maxiter': self.maxiter},
            'BFGS':      {'maxiter': self.maxiter},
            'Newton-CG': {'maxiter': self.maxiter},
            'L-BFGS-B':  {'maxiter': self.maxiter},
            'TNC':       {'maxfun':  self.maxiter},
            'SLSQP':     {'maxiter': self.maxiter},
        }

        results = {}
        for method in methods:
            iters = []
            start = time.perf_counter()

            res = minimize(
                fun=self.loss,
                x0=w0,
                jac=self.gradient,
                hess=self.hessiana if method == 'Newton-CG' else None,
                args=(X, y),
                method=method,
                callback=lambda w: iters.append(self.loss(w, X, y)),
                tol=self.tol,
                options= max_iter_options[method],

            )

            elapsed = time.perf_counter() - start
            results[method] = {
                'time':      elapsed,
                'loss':      res.fun,
                'iters':     len(iters),
                'converged': res.success,
                'history':   iters,
                'w':         res.x
            }

            print(f"{method:12s} | tempo: {elapsed:.4f}s | loss: {res.fun:.4f} | iterações: {len(iters)} | convergiu: {res.success}")

        return results

    def predict(self, X, w):
        return np.sign(X @ w)
    
    def avaliar(self, X_test, y_test, results):
        for method, data in results.items():
            y_pred = self.predict(X_test, data['w'])
            f1  = f1_score(y_test, y_pred)
            acc = accuracy_score(y_test, y_pred)
            print(f"  {method:12s} | f1: {f1:.4f} | acc: {acc:.4f} | convergiu: {data['converged']}")



In [22]:
methods = ['CG', 'BFGS', 'Newton-CG', 'L-BFGS-B']
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

# Inicialização aleatória dos pesos
w0 = np.random.randn(X_train.shape[1]) * 0.01

for C in C_values:
    print(f"\n--- C = {C} ---")
    svm = SVMoptimizer(C=C, tol=1e-5, maxiter=1000)
    results = svm.otimizar(X_train_, y_train_, w0, methods)
    svm.avaliar(X_test_, y_test_, results)



--- C = 0.001 ---
CG           | tempo: 0.0203s | loss: 0.2004 | iterações: 7 | convergiu: True
BFGS         | tempo: 0.0409s | loss: 0.2004 | iterações: 11 | convergiu: True
Newton-CG    | tempo: 0.0082s | loss: 0.2004 | iterações: 5 | convergiu: True
L-BFGS-B     | tempo: 0.0056s | loss: 0.2004 | iterações: 4 | convergiu: True
  CG           | f1: 0.9552 | acc: 0.9600 | convergiu: True
  BFGS         | f1: 0.9552 | acc: 0.9600 | convergiu: True
  Newton-CG    | f1: 0.9552 | acc: 0.9600 | convergiu: True
  L-BFGS-B     | f1: 0.9552 | acc: 0.9600 | convergiu: True

--- C = 0.01 ---
CG           | tempo: 0.0285s | loss: 0.9184 | iterações: 17 | convergiu: True
BFGS         | tempo: 0.1016s | loss: 0.9184 | iterações: 15 | convergiu: True
Newton-CG    | tempo: 0.0131s | loss: 0.9184 | iterações: 8 | convergiu: True
L-BFGS-B     | tempo: 0.0047s | loss: 0.9184 | iterações: 9 | convergiu: True
  CG           | f1: 0.9697 | acc: 0.9733 | convergiu: True
  BFGS         | f1: 0.9697 | acc: 0

In [23]:
# Inicialização com pesos zerados
w0 = np.zeros(X_train.shape[1])

for C in C_values:
    print(f"\n--- C = {C} ---")
    svm = SVMoptimizer(C=C, tol=1e-5, maxiter=1000)
    results = svm.otimizar(X_train_, y_train_, w0, methods)
    svm.avaliar(X_test_, y_test_, results)


--- C = 0.001 ---
CG           | tempo: 0.0145s | loss: 0.2004 | iterações: 6 | convergiu: True
BFGS         | tempo: 0.0136s | loss: 0.2004 | iterações: 11 | convergiu: True
Newton-CG    | tempo: 0.0076s | loss: 0.2004 | iterações: 5 | convergiu: True
L-BFGS-B     | tempo: 0.0033s | loss: 0.2004 | iterações: 4 | convergiu: True
  CG           | f1: 0.9552 | acc: 0.9600 | convergiu: True
  BFGS         | f1: 0.9552 | acc: 0.9600 | convergiu: True
  Newton-CG    | f1: 0.9552 | acc: 0.9600 | convergiu: True
  L-BFGS-B     | f1: 0.9552 | acc: 0.9600 | convergiu: True

--- C = 0.01 ---
CG           | tempo: 0.0236s | loss: 0.9184 | iterações: 18 | convergiu: True
BFGS         | tempo: 0.0157s | loss: 0.9184 | iterações: 14 | convergiu: True
Newton-CG    | tempo: 0.0109s | loss: 0.9184 | iterações: 8 | convergiu: True
L-BFGS-B     | tempo: 0.0055s | loss: 0.9184 | iterações: 10 | convergiu: True
  CG           | f1: 0.9697 | acc: 0.9733 | convergiu: True
  BFGS         | f1: 0.9697 | acc: 

Os resultados se mantêm equivalentes para ambas as inicializações porque a squared hinge loss é uma função convexa e isso garante um único mínimo global, independente do ponto de partida. A qualidade das métricas (f1 ≈ 0.95) é explicada pela boa separabilidade linear dos dados

## 4.4 Adaboost

Formalmente, o AdaBoost tenta minimizar a seguinte função objetivo:

$$
L = \sum_{i=1}^{n} \exp(-y_i F(x_i))
$$

onde:

- $y_i \in \{-1, +1\}$ é o rótulo verdadeiro  
- $F(x_i)$ é o modelo final (combinação dos *weak learners*)  

O modelo final é dado por:

$$
F(x) = \sum_{t=1}^{T} \alpha_t h_t(x)
$$

onde:

- $h_t(x)$ são os classificadores fracos  
- $\alpha_t$ são os pesos de cada classificador

In [24]:
modelo = AdaBoostClassifier(algorithm='SAMME')

previsoes, pipeline = Modelagem(X_train, y_train, X_test, modelo)

print("=== Relatório de Métricas (Adaboost) ===")

print(classification_report(y_test, previsoes, target_names=['0', '1']))

=== Relatório de Métricas (Adaboost) ===
              precision    recall  f1-score   support

           0       0.89      0.95      0.92        43
           1       0.93      0.84      0.89        32

    accuracy                           0.91        75
   macro avg       0.91      0.90      0.90        75
weighted avg       0.91      0.91      0.91        75



### 4.4.1 Mudando os Métodos de Otimização

#### 4.4.1.1 Extraindo o Adaboost Treinado no Pipeline

In [25]:
ada = pipeline.named_steps['modelo']

#### 4.4.1.2 Aplicando scaler e PCA no treino

In [26]:
pca_saida = pipeline[:-1].transform(X_train)

#### 4.4.1.3 Pegando o número de estimadores base treinados pelo AdaBoost

In [27]:
n_est = len(ada.estimators_)

#### 4.4.1.4 Montando Matriz de Predições

Cada estimador do AdaBoost vota numa classe (-1 ou 1). O minimize precisa dessas predições organizadas em forma de matriz para calcular a loss e o gradiente. Aí quando você faz H @ alphas, está calculando o voto ponderado de todos os estimadores para cada amostra

In [28]:
H = np.zeros((X_train.shape[0], n_est))

for t, estimador in enumerate(ada.estimators_):
    H[:, t] = estimador.predict(pca_saida)

#### 4.4.1.5 Convertendo Labels 0/1 Para -1/1 

In [29]:
y_train_signed = np.where(y_train == 0, -1, 1)

#### 4.4.1.6 Funções Usadas No Minimize

In [30]:
def loss_exp(alphas, H, y):
    
    margem = np.clip(H @ alphas, -500, 500)
    
    return np.mean(np.exp(-y * margem))


def grad_loss(alphas, H, y):
    
    margem = np.clip(H @ alphas, -500, 500)
    
    pesos = -y * np.exp(-y * margem)  
    
    return (H.T @ pesos) / len(y)    

#### 4.4.1.7 Criando H_test Para Comparação Entre Alfas Obtidos

In [31]:
pca_saida_test = pipeline[:-1].transform(X_test)

H_test = np.zeros((X_test.shape[0], n_est))

for t, estimador in enumerate(ada.estimators_):
    H_test[:, t] = estimador.predict(pca_saida_test)

In [32]:
methods = {
    'CG':        None,   # gradiente conjugado
    'BFGS':      None,   # quase-Newton denso
    'Newton-CG': None,   # Newton com gradiente conjugado interno
    'L-BFGS-B':  None    # quase-Newton com memória limitada
}

historicos = {}


for met, bounds in methods.items():
    historico = []
    
    def callback(alphas):
        historico.append(alphas.copy())
        
    result = minimize(
        loss_exp,
        x0=ada.estimator_weights_,  # chute inicial
        jac=grad_loss,              # gradiente analítico 
        args=(H, y_train_signed),
        method=met,
        bounds=bounds,
        callback=callback
    )
    
    historicos[met] = np.array(historico)
    alphas_otimizados = result.x

    # combinação linear dos estimadores com alfas otimizados
    previsoes_otimizadas = np.sign(H_test @ alphas_otimizados)

    # converte de volta para 0/1 (de -1/1)
    previsoes_otimizadas = np.where(previsoes_otimizadas == -1, 0, 1).astype(int)

    print(f"Método:     {met}")
    print(f"Convergiu:  {result.success}")
    print(f"Loss final: {result.fun:.6f}")
    print(f"Iterações:  {result.nit}")
    print(f"Acurácia:   {accuracy_score(y_test, previsoes_otimizadas):.4f}")
    print("----------------")

Método:     CG
Convergiu:  True
Loss final: 0.337099
Iterações:  689
Acurácia:   0.8667
----------------
Método:     BFGS
Convergiu:  True
Loss final: 0.337064
Iterações:  380
Acurácia:   0.8667
----------------
Método:     Newton-CG
Convergiu:  False
Loss final: 0.337069
Iterações:  24
Acurácia:   0.8667
----------------
Método:     L-BFGS-B
Convergiu:  True
Loss final: 0.337065
Iterações:  233
Acurácia:   0.8667
----------------


## 4.5 - K-Nearest Neighbors (KNN)

## Função de decisão do KNN

O KNN classifica uma amostra $x$ com base nos $k$ vizinhos mais próximos no espaço de features.
A classe predita é dada por:

$$\hat{y} = \arg\max_{c} \sum_{i=1}^{k} \mathbf{1}[y_i = c], \quad x_i \in \mathcal{N}_k(x)$$

onde:

- $\mathcal{N}_k(x)$ é o conjunto dos $k$ vizinhos mais próximos de $x$
- $\mathbf{1}[\cdot]$ é a função indicadora (vale 1 se a condição for verdadeira, 0 caso contrário)
- $c$ são as classes possíveis
- $k$ é o número de vizinhos definido pelo usuário



## Distância de Minkowski

Para encontrar os vizinhos mais próximos, o KNN utiliza a **distância de Minkowski**:

$$d(x, x_i) = \left( \sum_{j=1}^{p} |x_j - x_{ij}|^p \right)^{1/p}$$

onde:

- $x_j$ é o $j$-ésimo atributo da amostra de consulta
- $x_{ij}$ é o $j$-ésimo atributo do ponto de treino $x_i$
- $p$ controla o tipo de distância:
  - $p = 1$ → Distância de **Manhattan**
  - $p = 2$ → Distância **Euclidiana**


In [34]:
modelo = KNeighborsClassifier()

previsoes, pipeline = Modelagem(X_train, y_train, X_test, modelo)

print("=== Relatório de Métricas (KNN) ===")

print(classification_report(y_test, previsoes, target_names=['0', '1']))

=== Relatório de Métricas (KNN) ===
              precision    recall  f1-score   support

           0       0.89      0.98      0.93        43
           1       0.96      0.84      0.90        32

    accuracy                           0.92        75
   macro avg       0.93      0.91      0.92        75
weighted avg       0.92      0.92      0.92        75

